# 01 — Dataset Exploration

**FairLens AI / NyayaLens — Person 3 (ML Layer)**

This notebook loads the UCI Adult (Census Income) dataset and explores its
structure, distributions, and sensitive attributes before any modelling.

### What we do here
1. Load the Adult dataset from OpenML
2. Inspect shape, columns, dtypes
3. Check for missing values and `?` placeholders
4. Explore the target variable distribution
5. Explore sensitive attribute distributions (sex, race)
6. Cross-tabulate target × sensitive attributes to see raw disparities

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml

sns.set_theme(style="whitegrid")
print("Libraries loaded.")

## Step 1 — Load the dataset

`fetch_openml` downloads the Adult dataset (also called "Census Income") from
OpenML. It has ~48,842 rows and 15 columns. The target column is named `class`
and contains `>50K` or `<=50K`.

In [ ]:
adult_bunch = fetch_openml("adult", version=2, as_frame=True)
raw_df = adult_bunch.frame

print(f"Shape: {raw_df.shape}")
print(f"Columns: {list(raw_df.columns)}")
raw_df.head()

## Step 2 — Data types and basic info

In [ ]:
raw_df.info()

In [ ]:
raw_df.describe(include="all")

## Step 3 — Missing values

Some columns store `?` as a string for missing values. Let's count them.

In [ ]:
# Count NaN values per column
nan_counts = raw_df.isna().sum()
print("NaN counts per column:")
print(nan_counts[nan_counts > 0])
print()

# Count '?' values in string columns
question_mark_counts = {}
for col in raw_df.columns:
    if raw_df[col].dtype == object:
        count = (raw_df[col] == "?").sum()
        if count > 0:
            question_mark_counts[col] = count

print("'?' counts per column:")
for col, count in question_mark_counts.items():
    print(f"  {col}: {count}")

total_rows = raw_df.shape[0]
total_bad = sum(question_mark_counts.values()) + nan_counts.sum()
print(f"\nTotal rows: {total_rows}")
print(f"Rows we'll lose after cleaning: ~{total_bad} (some overlap)")

## Step 4 — Target variable distribution

The target `class` tells us whether a person earns >50K or <=50K per year.
This dataset is imbalanced — most people earn <=50K.

In [ ]:
target_counts = raw_df["class"].value_counts()
print("Target distribution:")
print(target_counts)
print(f"\nPositive rate (>50K): {target_counts.get('>50K', 0) / total_rows:.2%}")

fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind="bar", ax=ax, color=["#4CAF50", "#F44336"])
ax.set_title("Target Distribution: Income Class")
ax.set_ylabel("Count")
ax.set_xlabel("Income Class")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 5 — Sensitive attribute distributions

We look at `sex` and `race` — the columns we'll use for fairness analysis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

raw_df["sex"].value_counts().plot(kind="bar", ax=axes[0], color=["#2196F3", "#FF9800"])
axes[0].set_title("Distribution: Sex")
axes[0].set_ylabel("Count")
plt.sca(axes[0])
plt.xticks(rotation=0)

raw_df["race"].value_counts().plot(kind="bar", ax=axes[1], color=sns.color_palette("Set2"))
axes[1].set_title("Distribution: Race")
axes[1].set_ylabel("Count")
plt.sca(axes[1])
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

print("Sex counts:")
print(raw_df["sex"].value_counts())
print("\nRace counts:")
print(raw_df["race"].value_counts())

## Step 6 — Cross-tabulation: Target × Sensitive Attributes

This shows the raw income distribution broken down by sex. If there is a
large difference in the >50K rate between Male and Female, that's a signal
of **label imbalance** — the dataset itself is skewed, even before any model.

In [ ]:
cross_tab = pd.crosstab(raw_df["sex"], raw_df["class"], normalize="index")
print("Income distribution by sex (proportions):")
print(cross_tab)
print()

cross_tab.plot(kind="bar", stacked=True, figsize=(8, 4),
               color=["#4CAF50", "#F44336"])
plt.title("Income Distribution by Sex")
plt.ylabel("Proportion")
plt.xlabel("Sex")
plt.xticks(rotation=0)
plt.legend(title="Income")
plt.tight_layout()
plt.show()

In [ ]:
cross_tab_race = pd.crosstab(raw_df["race"], raw_df["class"], normalize="index")
print("Income distribution by race (proportions):")
print(cross_tab_race)
print()

cross_tab_race.plot(kind="bar", stacked=True, figsize=(10, 4),
                    color=["#4CAF50", "#F44336"])
plt.title("Income Distribution by Race")
plt.ylabel("Proportion")
plt.xlabel("Race")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Income")
plt.tight_layout()
plt.show()

## Summary

- The dataset has ~48K rows with 15 columns.
- About 24% of rows are positive class (>50K).
- The dataset is heavily **Male-dominated** (~2:1 ratio).
- Males have a much higher rate of >50K income than Females in the raw data.
- This label imbalance is exactly what our bias detection pipeline will measure.

**Next:** Notebook 02 — Baseline Models